# EXP-037 · Progressive vision–language adaptation

Attach the private `exp037_inputs.zip` dataset, switch on **GPU** and **Internet**, set
`AUTHORIZE_RUN = True`, and select **Run All**. Use one GPU, preferably an L4 or A100 if offered.
The notebook also supports Kaggle's T4 accelerator and uses only the first GPU.

The run verifies the package, sets up the pinned Python/CUDA runtime, then trains A (4 epochs),
B (6), and C (8). Each stage starts with a real-image compatibility and timing check. Each
completed stage exports its selected standalone and incumbent blend. Both use public Development
for selection. Qwen's eligibility under the ST3 rules is still unconfirmed.

The first session can take up to 10.5 hours including setup. If more time is needed, save and
download the latest results or progress ZIP and its SHA-256 receipt from **Output**. In a fresh
session on the same GPU type, attach the original dataset and the previous ZIP as private data.
Set `RESUME_INPUT` to the ZIP or the Kaggle-extracted folder containing `export_manifest.json`.

In [ ]:
AUTHORIZE_RUN = False
RESUME_INPUT = ""      # Optional absolute path to the previous progress or results ZIP, or its extracted root
SIGLIP_SNAPSHOT = ""   # Optional absolute path to the pinned model snapshot
QWEN_SNAPSHOT = ""     # Optional absolute path to the pinned model snapshot

In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import subprocess
import sys
import time
from zipfile import ZipFile

WORKER_SOURCE = "38b1e381127c5f4fdc02dfc9845caeba90d9667e"
MANIFEST_SHA256 = "026720f406c02236aeaf5bfdc806853816323b66267f81f0e3e2dbbfe1ce7e0e"
ARCHIVE_SHA256 = "eaf16b9301cb131fbdf61a1b86fb191d3617b2141e39398f0c3b12bcf57ffce2"

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def verify_root(root, manifest_sha, source):
    manifest_path = root / "exp037_manifest.json"
    if sha256(manifest_path) != manifest_sha:
        raise ValueError(f"Input manifest checksum differs at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest["source"] != source:
        raise ValueError("The private dataset was prepared for a different worker revision")
    for name, expected in manifest["files"].items():
        relative = PurePosixPath(name)
        target = root / relative
        if relative.is_absolute() or ".." in relative.parts or not target.is_file() or sha256(target) != expected:
            raise ValueError(f"Missing or corrupt expected file: {target}. Reattach the verified EXP-037 dataset.")
    return manifest

def locate_inputs(mount, work, manifest_sha, archive_sha, source):
    discovered = sorted(mount.rglob("exp037_manifest.json"))
    matches = [path.parent for path in discovered if sha256(path) == manifest_sha and (path.parent / "runtime").is_dir()]
    if len(matches) > 1:
        raise ValueError("Multiple extracted copies match. Attach only one initial EXP-037 dataset.")
    if matches:
        root = matches[0]
    else:
        archives = sorted(mount.rglob("exp037_inputs.zip"))
        matches = [path for path in archives if sha256(path) == archive_sha]
        if len(matches) != 1:
            raise FileNotFoundError(
                "Attach exactly one private EXP-037 input dataset from this notebook's handoff. "
                f"Found {len(discovered)} manifests and {len(archives)} ZIP files, neither with one matching identity."
            )
        restored = work / manifest_sha[:16]
        restored.mkdir(parents=True, exist_ok=True)
        # The immutable archive hash verified above pins its validated member paths.
        with ZipFile(matches[0]) as archive:
            archive.extractall(restored)
        root = restored / "exp037_inputs"
    return root, verify_root(root, manifest_sha, source)

STARTED = time.time()
MOUNT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working")
if not AUTHORIZE_RUN:
    raise RuntimeError("Attach the private input, enable GPU and Internet, set AUTHORIZE_RUN = True and select Run All.")
if not MOUNT.is_dir() or not OUTPUT.is_dir():
    raise RuntimeError("Open this notebook in Kaggle with the private EXP-037 input attached.")
gpu_probe = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if Path("/dev/nvidia0").exists() else None
if gpu_probe is None or gpu_probe.returncode != 0 or "GPU 0" not in gpu_probe.stdout:
    raise RuntimeError("Select a CUDA GPU in Kaggle Settings, then Run All again.")
print(gpu_probe.stdout.strip())
input_root, manifest = locate_inputs(MOUNT, Path("/kaggle/temp/exp037_input"),
                                     MANIFEST_SHA256, ARCHIVE_SHA256, WORKER_SOURCE)
print("Verified", len(manifest["files"]), "files at", input_root)
print("Training and Development populations:", manifest["population"])

In [ ]:
launcher = input_root / "runtime/scripts/exp037_kaggle.py"
command = [sys.executable, "-u", str(launcher), "--inputs", str(input_root), "--handoff",
           MANIFEST_SHA256, "--started", str(STARTED)]
for option, value in (("resume", RESUME_INPUT), ("siglip", SIGLIP_SNAPSHOT), ("qwen", QWEN_SNAPSHOT)):
    if value:
        path = Path(value)
        if not path.is_absolute() or not path.exists():
            raise FileNotFoundError(f"Set {option.upper()} to its actual absolute Kaggle mount path: {path}")
        command.extend((f"--{option}", str(path)))
subprocess.run(command, check=True, env=os.environ | {"PYTHONDONTWRITEBYTECODE": "1"})

In [ ]:
run_dir = OUTPUT / "exp037_run"
if (run_dir / "status.json").is_file():
    print((run_dir / "status.json").read_text())
for name in ("exp037_progress.zip", "exp037_results.zip", "exp037_diagnostic.zip"):
    path = OUTPUT / name
    if path.is_file():
        print(path)
        checksum = path.with_suffix(".zip.sha256")
        if checksum.is_file():
            print(checksum.read_text().strip())
for stage in ("A", "B", "C"):
    receipt_path = run_dir / stage / "selected/summary.json"
    if receipt_path.is_file():
        report = json.loads(receipt_path.read_text())
        print(stage, "epoch", report["standalone"]["epoch"],
              "standalone", report["standalone"]["metrics"]["full"],
              "blend", report["ensemble"]["alpha"], report["ensemble"]["full"])
if (run_dir / "selection.json").is_file():
    print("Best Development candidate:", json.loads((run_dir / "selection.json").read_text())["winner"])

In **Output**, save or download the results archive and its checksum. While the run is live,
`exp037_progress.zip` is replaced atomically at regular intervals and at epoch and stage boundaries.
Completed stage archives are also available in `exp037_run/<stage>/selected/`.